In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

# Using _trajectory-container-tools_ with ROS Bags


## Import trajectory-container-tools namespace

In [ ]:
import trajectory_container_tools as tct

## Specify the rosbag path

**Requirements**

The ROS bag must contain valid ROS message topics with proper timestamps. Supported message types include
`Odometry`, `AckermannDriveStamped`, `TFMessage`, `LaserScan`, `Imu`, `VescImuStamped` and more, plus you can implement your own, its realy easy.


In [ ]:
BAG = "2024-03-21_14-52-35-filtered"
rosbag_path = os.path.join( "data", "repository_data", "tests_data", "rosbag_test_data", "rosbag-vaul-f110-grand-salon-raw-msg", BAG)

print(f"Using ROS bag: {rosbag_path}")

## Introspect the ros bag contents

#### Validate path and check rosbag available topics

In [ ]:
rosbag_path = tct.extractor.check_bag_topics(rosbag_path)

## 1. Basic ROS Bag Processing

To extract features from a ROS bag, set a valid ros2 rosbag path and specify which topic messages (i.e., the feature key) and which message fields to extract (i.e., the value key). The value key can be either a `TrajectoryDataclass` or a tuple definning a on the fly custom tct dataclass name and the field to extract.
More on this in section _Define which topic to extract_.

### Feature configuration example using a `TrajectoryDataclass`
Check `tct.dataclasses` namespace for available TCT dataclasses or implement your own, its fast and simple.

In [ ]:
from trajectory_container_tools.dataclasses import NavMsgsOdometry, SensorMsgsImu, AckermannMsgsAckermannDriveStamped

tc_odom = tct.extractor.from_rosbag(rosbag_path,dataset_info=f"Robot: f110_race_car, Track: grand_salon, Run: {BAG}",features_config={
        "/odom": NavMsgsOdometry,
    })
print(tc_odom)

### Feature configuration example with using a custom, on the fly, specification

In [ ]:
tc_custom_2d_imu = tct.extractor.from_rosbag(rosbag_path,dataset_info=f"Robot: f110_race_car, Track: grand_salon, Run: {BAG}",features_config={
        '/sensors/imu/raw':   ('SensorMsgsImu2D', 'linearAcceleration_x',
                                          'linearAcceleration_y',
                                          'angularVelocity_z')
    })
print(tc_custom_2d_imu)

### Define which topic to extract

The `features_config` specifies the ROS message types to extract from the bag and aggregate them in a `Multifeature` dataclass. Message types are specified using existing `RosBagFeatureDataclass` subclasses such as: `NavMsgsOdometry`, `SensorMsgsImu`, `Tf2MsgsTFMessage`, `AckermannMsgsAckermannDriveStamped` or by using tuple of strings for custom message structures.

The `features_config` parameter specify the feature name and type (i.e. topic name and type)
to lookout in the rosbag topic list and agregate them in a *multifeature* dataclass.

Feature dimensions such as `pose.position.x` or `twist.linear.y` are specified either by using existing `RosBagFeatureDataclass` subclass such as: `NavMsgsOdometry`, `AckermannMsgsAckermannDriveStamped`, `Tf2MsgsTFMessage`, `SensorMsgsImu` or by using tuple of strings such as `('<NewFeatureDataclassTypeName>', '<topic_property_name_1>', '<topic_property_name_2>', ...)` with `NewFeatureDataclassTypeName` being the ros topic type, in camelback notation, without the `/msg` directory e.g., `NavMsgsOdometry` -> `nav_msgs/msg/Odometry`.

```python
feature_config = {
    '/pf/pose/odom':        NavMsgsOdometry,
    '/odom':                NavMsgsOdometry,
    '/sensors/imu/raw':     ('SensorMsgsImu2D', 'linearAcceleration_x',
                                                'linearAcceleration_y',
                                                'angularVelocity_z')
}
```
Topic property name `drive_steeringAngleVelocity` would convert to ros topic msg `drive.steering_angle_velocity`. The parsing rule for topic property name is:
 - underscore `_` convert to dot `.`
 - `CamelCase` convert to `snake_case`.

💎 Note that each `RosBagFeatureDataclass` subclass validates that extracted data has proper timestamps and uniform shape.


### Timestamps causal ordering validation

On instanciation, the trajectory dataclass execute a timestamps validation to check that they are monoticaly increassing (i.e., timestamps are in causal order such that timastamp at t=0 < t+1 < ... < t=T) or throw a `TimestampCausalOrderingError` otherwise.

In [ ]:
BAG_OFFENDING_TS = "2024-03-21_14-52-35-offending-timestamps"
rosbag_offending_timestamp_path = os.path.join( "data", "repository_data", "tests_data", "rosbag_test_data", "rosbag-vaul-f110-grand-salon-raw-msg", BAG_OFFENDING_TS)
rosbag_offending_timestamp_path = tct.extractor.check_bag_topics(rosbag_offending_timestamp_path)

try:
    tct.extractor.from_rosbag(rosbag_offending_timestamp_path,dataset_info=f"Robot: f110_race_car, Track: grand_salon, Run: {BAG_OFFENDING_TS}",features_config={
            "/odom": NavMsgsOdometry,
            "/teleop": AckermannMsgsAckermannDriveStamped,
        })
except tct.TimestampCausalOrderingError as e:
    print("Detected timestamps causal ordering violation!", e)

**Solution**: chose an rosbag timestamp interval where there is no timestamps causal ordering violation

In [ ]:
tct.extractor.from_rosbag(rosbag_offending_timestamp_path,dataset_info=f"Robot: f110_race_car, Track: grand_salon, Run: {BAG_OFFENDING_TS}",features_config={
        "/odom": NavMsgsOdometry,
        "/teleop": AckermannMsgsAckermannDriveStamped,
    },start=1711047206000000000,stop=1711047237203288917)
print("Successfully extracted features!")

## 2. Accessing Data

ROS message data can be accessed through the structured dataclass attributes. Each message type provides access to its specific fields.

### You can access each features and their fields by property call

In [ ]:
print(type(tc_odom.topic_odom))
print(type(tc_odom.topic_odom.pose))
print(type(tc_odom.topic_odom.pose.pose))
print(type(tc_odom.topic_odom.pose.pose.position))
print(type(tc_odom.topic_odom.pose.pose.position.x))

### You can access each features and their dimension using attribute getter

In [ ]:
# Direct attribute call
id_direct = id(tc_odom.get_dynamic_field("topic_odom").pose.pose.position.x)

# Nested attribute call
id_nested = id(tc_odom.fetch_nested_attribute("topic_odom.pose.pose.position.x"))

assert id_direct == id_nested

## 3. Trajectory Visualization

Visualize the extracted trajectory data to understand robot motion patterns.


In [ ]:
from trajectory_container_tools.utils.plot import plot_trajectory_2d

plot_trajectory_2d(tc_odom)

## 4. Simplify your code and minimize risk of piping the wrong data

### Example: Velocity and Control Analysis

Analyze the robot's velocity profile and control characteristics.


In [ ]:
from typing import Union

def plot_velocity_analysis(trajectory_data: Union[tct.BaseTrajectoryDataclass], title: str = "Velocity Analysis"):
    """Plot velocity components and speed over time."""
    tc_odom_: NavMsgsOdometry = trajectory_data.topic_odom

    # Calculate speed
    _speed = np.sqrt(tc_odom_.twist.twist.linear.x**2 + tc_odom_.twist.twist.linear.y**2)

    fig, axes = plt.subplots(3, 1, figsize=(12, 7))

    # Linear velocity components
    axes[0].plot(tc_odom_.timesteps_indices, tc_odom_.twist.twist.linear.x, 'b-', label='Velocity X', linewidth=2)
    axes[0].plot(tc_odom_.timesteps_indices, tc_odom_.twist.twist.linear.y, 'r-', label='Velocity Y', linewidth=2)
    axes[0].plot(tc_odom_.timesteps_indices, _speed, 'k--', label='Speed', linewidth=2)
    axes[0].set_ylabel('Linear Velocity (m/s)')
    axes[0].set_title(f'{title} - Linear Velocity')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Angular velocity
    axes[1].plot(tc_odom_.timesteps_indices, tc_odom_.twist.twist.angular.z, 'g-', linewidth=2)
    axes[1].set_ylabel('Angular Velocity (rad/s)')
    axes[1].set_title('Angular Velocity Z')
    axes[1].grid(True, alpha=0.3)

    # Speed histogram
    axes[2].hist(_speed, bins=30, alpha=0.7, color='purple', edgecolor='black')
    axes[2].set_xlabel('Speed (m/s)')
    axes[2].set_ylabel('Frequency')
    axes[2].set_title('Speed Distribution')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    return fig, axes

# Analyze velocity profile
plot_velocity_analysis(tc_odom, tc_odom.dataset_info)
plt.show()

# Print velocity statistics
vel_x = tc_odom.topic_odom.twist.twist.linear.x
vel_y = tc_odom.topic_odom.twist.twist.linear.y
speed = np.sqrt(vel_x**2 + vel_y**2)
angular_z = tc_odom.topic_odom.twist.twist.angular.z

print(f"\nVelocity Statistics:")
print(f"Max speed: {np.max(speed):.3f} m/s")
print(f"Mean speed: {np.mean(speed):.3f} m/s")
print(f"Max angular velocity: {np.max(np.abs(angular_z)):.3f} rad/s ({np.degrees(np.max(np.abs(angular_z))):.1f} deg/s)")

## 5. Advanced Processing with Custom Post-Processing Callback

Create custom feature classes with specialized post-processing

### Example for racing data analysis

#### 1. Extend `NavMsgsOdometry` with `post_init_feature_callback` override


In [ ]:
@dataclass
class RacingOdometryAnalysis(NavMsgsOdometry):
    """Custom odometry analysis for racing applications."""

    def post_init_feature_callback(self, feature_name):
        """Calculate racing-specific metrics."""

        if feature_name == "pose":
            # Calculate trajectory metrics when position data is processed
            x = self.fetch_nested_attribute("pose.pose.position.x")
            y = self.fetch_nested_attribute("pose.pose.position.y")

            # Calculate path length
            dx = np.diff(x)
            dy = np.diff(y)
            segment_lengths = np.sqrt(dx**2 + dy**2)
            path_length = np.sum(segment_lengths)
            self.set_dynamic_field('path_length', path_length)

        elif feature_name == "twist":
            # Calculate acceleration when velocity data is processed
            _vel_x = self.fetch_nested_attribute("twist.twist.linear.x")
            _vel_y = self.fetch_nested_attribute("twist.twist.linear.y")

            # Calculate speed and acceleration
            _speed = np.sqrt(_vel_x**2 + _vel_y**2)
            self.set_dynamic_field('speed', _speed)

            if len(self.header.timestamps) > 1:
                dt = self.header.timestamps.delta_stamps
                acceleration = np.diff(_speed) / dt[1:]
                # Pad to match length
                self.set_dynamic_field('acceleration', np.concatenate([[0], acceleration]))
            else:
                self.set_dynamic_field('acceleration', np.zeros_like(_speed))

        return None

#### 2. Multi-Feature ROS Bag Analysis

Process multiple ROS message types simultaneously for comprehensive robot behavior analysis.


In [ ]:
multi_features_config = {
    "/odom": RacingOdometryAnalysis,
    "/sensors/imu/raw": SensorMsgsImu,
    "/teleop": AckermannMsgsAckermannDriveStamped,
}

multi_data = tct.extractor.from_rosbag(rosbag_path,dataset_info="Multi-sensor F110 data analysis",features_config=multi_features_config,start=None,stop=None)

multi_data.summary

In [ ]:
print(f"\nRacing Analysis Results:")
print(f"  Total path length: {multi_data.topic_odom.path_length} meters")
print(f"  Maximum speed: {np.max(multi_data.topic_odom.speed):.2f} m/s")
print(f"  Maximum acceleration: {np.max(np.abs(multi_data.topic_odom.acceleration)):.2f} m/s²")

#### 3. Use the new available fields in visualitation

Visualize racing-specific metrics including speed profiles, acceleration, and track curvature.


In [ ]:
def plot_racing_analysis(racing_data, title: str = "Racing Performance Analysis"):
    """Plot comprehensive racing analysis."""
    odom = racing_data.topic_odom

    fig = plt.figure(figsize=(15, 6))

    # Create a 2x3 grid of subplots
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 1])

    # 1. Trajectory with speed colormap
    ax1 = fig.add_subplot(gs[:, 0])
    x = odom.pose.pose.position.x
    y = odom.pose.pose.position.y
    _speed = odom.speed

    scatter = ax1.scatter(x, y, c=_speed, cmap='viridis', s=20, alpha=0.8)
    ax1.plot(x, y, 'k-', alpha=0.3, linewidth=1)
    ax1.scatter(x[0], y[0], color='green', s=200, marker='o',
               label='Start', edgecolor='black', linewidth=3, zorder=5)
    ax1.scatter(x[-1], y[-1], color='red', s=200, marker='s',
               label='End', edgecolor='black', linewidth=3, zorder=5)

    cbar = plt.colorbar(scatter,)
    cbar.set_label('Speed (m/s)', fontsize=12)
    ax1.set_xlabel('X Position (m)')
    ax1.set_ylabel('Y Position (m)')
    ax1.set_title(f'{title} - Speed Profile on Track')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_aspect('equal')

    # 2. Speed vs time
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(odom.timesteps_indices, _speed, 'b-', linewidth=2)
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Speed (m/s)')
    ax2.set_title('Speed vs Time')
    ax2.grid(True, alpha=0.3)

    # 3. Acceleration vs time
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.plot(odom.timesteps_indices, odom.acceleration, 'r-', linewidth=2)
    ax3.axhline(y=0, color='k', linestyle='--', alpha=0.5)
    ax3.set_xlabel('Time (s)')
    ax3.set_ylabel('Acceleration (m/s²)')
    ax3.set_title('Acceleration vs Time')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    return fig

# Create racing analysis visualization
plot_racing_analysis(multi_data, multi_data.dataset_info)
plt.show()

## Summary

This notebook demonstrated comprehensive ROS bag processing with Trajectory Container Tools:

1. **Basic Feature Extraction**: Loading and processing ROS bag data
2. **Data Access**: Accessing structured ROS message data
3. **Visualization**: Creating trajectory and velocity plots
4. **Custom Processing**: Implementing racing-specific analysis
5. **Multi-Feature Analysis**: Processing multiple ROS message types

### Key Features Demonstrated:

- **ROS Message Support**: `nav_msgs/Odometry`, `tf2_msgs/TFMessage`, `sensor_msgs/Imu`
- **Automatic Validation**: Timestamp synchronization and data consistency
- **Custom Post-Processing**: Domain-specific analysis (racing metrics)
- **Comprehensive Visualization**: Trajectory, speed, acceleration, and curvature analysis
